# Gemini 모델별 비교 — 도입부 복습·연결 문장 추출

비교 모델: `gemini-2.0-flash` / `gemini-2.5-pro` / `gemini-3.5-flash` / `gemini-3.1-pro-preview`

**태스크**: `lectures_kss.csv` 에서 하나의 강의 도입부(첫 30분)를 읽고,  
전날 복습을 간략히 언급하면서 오늘 내용과 연결하는 문장을 추출한다.

In [2]:
!pip install google.generativeai

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 29.0 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 30.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [google.generativeai]ogle-ai-generativelanguage]


In [11]:
import sys, csv, asyncio
from pathlib import Path
from IPython.display import display, Markdown

sys.path.insert(0, '..')
from app.core.config import Settings

settings = Settings(_env_file='../.env')
assert settings.api_key, ".env 에 API_KEY 가 없습니다."

import google.generativeai as genai

genai.configure(api_key=settings.api_key)
print("Gemini client OK")

Gemini client OK


In [12]:
MODELS = [
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "gemini-3.5-flash",
    "gemini-3.1-pro-preview",
]

KSS_PATH  = Path("../data/processed/lectures_kss.csv")
INTRO_MIN = 30  # 도입부 기준: 강의 시작 후 N분

## 1. 데이터 로드 — 도입 30분 텍스트 추출

In [13]:
def load_intro_text(path: Path, date: str | None = None, minutes: int = 30) -> dict:
    """kss csv 에서 특정 날짜 강의의 첫 N분 발화를 합쳐 반환한다."""
    rows = list(csv.DictReader(path.open(encoding='utf-8')))

    if date is None:
        date = sorted({r['date'] for r in rows})[0]

    date_rows = [r for r in rows if r['date'] == date]

    def to_sec(ts: str) -> int:
        h, m, s = map(int, ts.split(':'))
        return h * 3600 + m * 60 + s

    start_sec  = to_sec(date_rows[0]['timestamp'])
    cutoff_sec = start_sec + minutes * 60

    intro = [
        r for r in date_rows
        if start_sec <= to_sec(r['timestamp']) <= cutoff_sec
    ]

    # 타임스탬프를 각 줄에 붙인 형식 (모델에 전달용)
    timestamped_lines = '\n'.join(
        f"[{r['timestamp']}] {r['text_raw']}" for r in intro
    )

    return {
        'date'             : date,
        'lecture_id'       : intro[0]['lecture_id'],
        't_start'          : intro[0]['timestamp'],
        't_end'            : intro[-1]['timestamp'],
        'row_count'        : len(intro),
        'text'             : ' '.join(r['text_raw'] for r in intro),
        'timestamped_lines': timestamped_lines,
    }

# ── 실행 ─────────────────────────────────────────────────────────
intro = load_intro_text(KSS_PATH, date="2026-02-02")

print(f"강의 ID : {intro['lecture_id']}")
print(f"날짜    : {intro['date']}")
print(f"구간    : {intro['t_start']} ~ {intro['t_end']} ({intro['row_count']}문장)")
print()
print("--- timestamped_lines 미리보기 (앞 5줄) ---")
print('\n'.join(intro['timestamped_lines'].splitlines()[:5]))

강의 ID : kdt-backendj-21th
날짜    : 2026-02-02
구간    : 09:11:17 ~ 09:40:46 (192문장)

--- timestamped_lines 미리보기 (앞 5줄) ---
[09:11:17] 여러분 오늘 수업 진행하도록 하겠습니다.
[09:11:17] 저희가 이제 오늘 1차 잡바 마지막 날입니다.
[09:11:17] 그래서 지난 시간에 제너릭 타입을 이용을 해서 커스텀 컬렉션을 활용한 크루드 방법을 학습 이해를 하고 그다음에 데이터 흐름 읽기 쓰기를 구현하는 날이에요.
[09:11:17] 자바는 NIO 패키지가 있고 그 다음에 NIO2라는 패키지를 가지고 있는데, 이 NI2는 자바 하는 애들이에요.
[09:11:18] 그래서 이걸 세 가지 섹션으로 나눠서 진행을 하고 있습니다.


## 2. 프롬프트 작성

In [49]:
PROMPT_TEMPLATE = """\
당신은 강의 전사 텍스트를 분석하는 교육 전문가입니다.

아래는 백엔드 개발 입문 강의의 도입부(시작 후 약 30분) 전사 텍스트입니다.
각 줄은 [HH:MM:SS] 타임스탬프와 발화 내용으로 구성됩니다.
이 강의는 여러 날에 걸쳐 이어지는 시리즈 강의입니다.

[도입부 전사 텍스트]
{timestamped_lines}

---

위 텍스트에서 **전날 학습 내용을 간략히 복습하면서 오늘 강의 내용과 연결하는 문장(들)**을 추출하세요.

추출 기준:
- "지난 시간에", "저번에", "어제", "앞에서" 등 이전 수업을 언급하는 표현이 포함된 문장
- 이전 내용을 상기시킨 뒤 오늘 학습할 주제나 목표로 이어지는 흐름
- 단순 인사말·출석 체크·행정 안내는 제외

[중요 규칙]
- 문장과 타임스탬프는 반드시 위 전사 텍스트에 존재하는 것만 사용할 것
- 문장은 원문 그대로 복사할 것 — 요약·수정·재구성 금지
- 타임스탬프는 해당 문장이 실제로 등장하는 줄의 [HH:MM:SS]를 그대로 사용할 것
- 위 텍스트에 없는 문장이나 타임스탬프를 만들어내지 말 것

응답 형식:
**추출 문장 목록**
1. [HH:MM:SS] 문장 원문
2. [HH:MM:SS] 문장 원문
...

**연결 흐름 요약** (2~3줄 — 전날 어떤 내용을 복습했고 오늘 무엇으로 이어지는지)

해당하는 문장이 없으면 "복습·연결 문장 없음"이라고만 답하세요.
"""

prompt = PROMPT_TEMPLATE.format(timestamped_lines=intro['timestamped_lines'])
print(f"프롬프트 길이: {len(prompt)}자")

프롬프트 길이: 13540자


## 3. 모델 호출 (비동기 병렬)

In [14]:
import time

async def call_model(model_name: str, prompt: str) -> dict:
    t0 = time.time()
    try:
        model = genai.GenerativeModel(model_name)
        response = await asyncio.to_thread(model.generate_content, prompt)
        elapsed = time.time() - t0
        return {
            'model'  : model_name,
            'text'   : response.text,
            'elapsed': elapsed,
            'error'  : None,
        }
    except Exception as e:
        elapsed = time.time() - t0
        return {
            'model'  : model_name,
            'text'   : None,
            'elapsed': elapsed,
            'error'  : str(e),
        }

async def run_all(models, prompt):
    tasks = [call_model(m, prompt) for m in models]
    return await asyncio.gather(*tasks)

results = await run_all(MODELS, prompt)

for r in results:
    status = f"OK ({r['elapsed']:.1f}s)" if not r['error'] else f"ERROR — {r['error'][:80]}"
    print(f"[{r['model']}] {status}")

NameError: name 'prompt' is not defined

## 4. 결과 비교

In [51]:
display(Markdown(
    f"**분석 구간** | 날짜: `{intro['date']}` · "
    f"`{intro['t_start']} ~ {intro['t_end']}` ({intro['row_count']}문장)\n\n---"
))

for r in results:
    header = f"### {r['model']}  ({r['elapsed']:.1f}s)"
    if r['error']:
        body = f"❌ **오류**: `{r['error']}`"
    else:
        body = r['text']
    display(Markdown(f"{header}\n\n{body}\n\n---"))

**분석 구간** | 날짜: `2026-02-02` · `09:11:17 ~ 09:40:46` (192문장)

---

### gemini-2.5-flash  (11.5s)

**추출 문장 목록**
1. [09:11:17] 그래서 지난 시간에 제너릭 타입을 이용을 해서 커스텀 컬렉션을 활용한 크루드 방법을 학습 이해를 하고 그다음에 데이터 흐름 읽기 쓰기를 구현하는 날이에요.

**연결 흐름 요약**
지난 시간에는 제너릭 타입과 커스텀 컬렉션을 활용한 CRUD 방법을 학습했습니다. 오늘은 이어서 데이터 흐름을 읽고 쓰는 방법을 구현하며, 자바 IO 및 NIO 패키지를 다룰 예정입니다.

---

### gemini-2.5-pro  (19.3s)

**추출 문장 목록**
1. [09:11:17] 그래서 지난 시간에 제너릭 타입을 이용을 해서 커스텀 컬렉션을 활용한 크루드 방법을 학습 이해를 하고 그다음에 데이터 흐름 읽기 쓰기를 구현하는 날이에요.
2. [09:22:25] 여기 보면 얘가 지금 1번이 코어 라이브러리 그 다음에 이제 사부작사부작 내려서 컬렉션 프레임워크이라는 걸 눌러 우리가 이제 예전 어제 금요일 날이죠.

**연결 흐름 요약**
강사는 '지난 시간'에 학습한 '제네릭 타입을 활용한 커스텀 컬렉션 CRUD'를 언급하며 오늘 배울 '데이터 흐름(IO)'과 직접적으로 연결합니다. 또한, 참고 자료를 살펴보는 과정에서 이전에 다룬 '컬렉션 프레임워크'를 상기시키며 학습 내용의 연계성을 강조하고 있습니다.

---

### gemini-3.5-flash  (7.1s)

**추출 문장 목록**
1. [09:11:17] 그래서 지난 시간에 제너릭 타입을 이용을 해서 커스텀 컬렉션을 활용한 크루드 방법을 학습 이해를 하고 그다음에 데이터 흐름 읽기 쓰기를 구현하는 날이에요.

**연결 흐름 요약**
지난 수업에서 학습한 제네릭 타입을 활용한 커스텀 컬렉션 CRUD 구현 내용을 복습하고, 이를 바탕으로 오늘 배울 핵심 주제인 자바 IO/NIO 패키지를 활용한 데이터 흐름 읽기와 쓰기(입출력) 구현으로 자연스럽게 학습 내용을 연결하고 있습니다.

---

### gemini-3.1-pro-preview  (12.3s)

**추출 문장 목록**
1. [09:11:17] 그래서 지난 시간에 제너릭 타입을 이용을 해서 커스텀 컬렉션을 활용한 크루드 방법을 학습 이해를 하고 그다음에 데이터 흐름 읽기 쓰기를 구현하는 날이에요.

**연결 흐름 요약**
강사는 지난 시간에 다루었던 제너릭 타입과 커스텀 컬렉션을 활용한 CRUD(생성·조회·수정·삭제) 학습 내용을 간략하게 상기시킵니다. 이를 바탕으로 오늘 수업의 핵심 주제인 데이터 흐름 읽기 및 쓰기(Java IO, NIO 등)를 구현할 것임을 알리며 자연스럽게 오늘의 학습 목표로 연결하고 있습니다.

---

In [45]:
display(Markdown(
    f"**분석 구간** | 날짜: `{intro['date']}` · "
    f"`{intro['t_start']} ~ {intro['t_end']}` ({intro['row_count']}문장)\n\n---"
))

for r in results:
    header = f"### {r['model']}  ({r['elapsed']:.1f}s)"
    if r['error']:
        body = f"❌ **오류**: `{r['error']}`"
    else:
        body = r['text']
    display(Markdown(f"{header}\n\n{body}\n\n---"))

**분석 구간** | 날짜: `2026-02-03` · `09:11:04 ~ 09:40:57` (102문장)

---

### gemini-2.5-flash  (36.4s)

**추출 문장 목록**
1. [09:12:15] 저희 어제 액션 11번의 노티스 파일에 내용은 적었지만 그래도 스트링 기반이고 얘네들은 데이터가 순차 이동이에요.
2. [09:15:05] 그런데 어제 저희는 네트워크를 하지 않았으니까 파일이 입출력 단위만 지금 학습을 하셨는데 바이트 스트림 단위 바이트 단위의 바이트 스트림 자체가 모든 팔 유형을 아우를 수가 있다고 생각하시면 되고 그래서 NIO2에서는 파일즈 가 가지고 있는 뉴 아풋스트림 그 다음에 뉴 인풋스트림 이렇게 되면 바이트 단위로 사용할 수 있는 요서드이죠.
3. [09:15:22] 자바가 가지고 있는 I이O 패키지는 클래스를 선택을 데이터 유형에 따라서 사용하는 거고, 그다음에 NIO2 같은 경우에는 메서드를 대상으로 저희가 데이터 프로세싱을 선택적으로 사용을 하는 그런 구문을 어제 확인을하셨죠.
4. [09:15:57] 문자 스트림을 어제 가만히 생각해보면 코드가 바이트 단위는 캐릭터 셋을 지정하는 것을 둔감합니다.
5. [09:17:36] 어제 너무 많이 막 했지만, 실제 그냥 일반적인 파일 입출력이라든지.
6. [09:22:38] 저희 어제 이 메서드를 호출을 해서 디렉토리 탐색하는 거를 하셨고요.
7. [09:23:58] 이 네 가지를 저희가 어제 탐색을 하고 이 2가지 인터페이스를 구현해주는 핸들링 이벤트 핸들링이에요.
8. [09:24:41] 그런데 속성 같은 경우에는 어제도 이제트리뷰트를 가지고 있는 인터페이스가 따로 있었죠.
9. [09:25:08] 어제 굳이 4번 해서 파일을 한 4개를 만들어 가지고 이벤트 핸들링 하는 방법을 기존에 OP에 쓸 때 사용했던 문법과 접목을 해서 1번은 인터페이스 상속받고 하나는 인터페이스를 상속받은 클래스 구형 객체를 익스텐즈를 세 번째는 내가 또 커스텀 클래스를 만들어 가지고 익스텐즈를 했고 네 번째는 얘가 가지고 있는 자체를 후성 클래스 객체인 심플팔 디지털을 이용을 해서 사용을 했고 그거를 어제 했던 내용을 한번 정리를 해보면 1번 오버로드 메소드를 통해서 디렉토리 순환 결과 이벤트 파일을 확인하는 방법이 얘입니다.
10. [09:25:49] 파일비지털 인터페이스를 사용한 커스텀 파일비지털을 생성하는 방법을 어제 학습하셨습니다.
11. [09:26:37] 제어를 이렇게 할 수 있는 메소드들을 제공을 해줘 포스트나 프리오더나 이렇게 해서 어제 뭐 비제터라고 하는 일단 서픽스 메소드를 이용을 해서 일단은 만들어 가지고 얘네가 제공을 해주고 페일도 나면 이제 제공해주고 전체 4개의 메소드를 오버라이드를 하는 객체로 만들어 가지고 하나는 인터페이스 하나는 밑에 있는 심플 파일 비지터 클래스가 구현을 해서 우리가 사용할 수 있게끔 만들어줬습니다.
12. [09:28:12] 베이직 파일 어트리뷰트스라고 하는 것은 이게 어제 뭐가 있었어요.

**연결 흐름 요약**
강의자는 전날 다룬 파일 입출력의 스트림 처리 방식과 Java IO 및 NIO2의 활용법, 그리고 디렉토리 탐색 및 이벤트 핸들링 기법에 대한 학습 내용을 상기시키고 있습니다. 이는 현재 진행 중인 IO 관련 심화 학습에 대한 배경 지식을 재확인하고, NIO2의 효율적인 파일 및 디렉토리 관리 기능을 오늘 강의의 주요 내용으로 자연스럽게 이어가는 역할을 합니다.

---

### gemini-2.5-pro  (27.7s)

**추출 문장 목록**
1. [09:12:15] 저희 어제 액션 11번의 노티스 파일에 내용은 적었지만 그래도 스트링 기반이고 얘네들은 데이터가 순차 이동이에요.
2. [09:15:05] 그런데 어제 저희는 네트워크를 하지 않았으니까 파일이 입출력 단위만 지금 학습을 하셨는데 바이트 스트림 단위 바이트 단위의 바이트 스트림 자체가 모든 팔 유형을 아우를 수가 있다고 생각하시면 되고 그래서 NIO2에서는 파일즈 가 가지고 있는 뉴 아풋스트림 그 다음에 뉴 인풋스트림 이렇게 되면 바이트 단위로 사용할 수 있는 요서드이죠.
3. [09:15:22] 자바가 가지고 있는 I이O 패키지는 클래스를 선택을 데이터 유형에 따라서 사용하는 거고, 그다음에 NIO2 같은 경우에는 메서드를 대상으로 저희가 데이터 프로세싱을 선택적으로 사용을 하는 그런 구문을 어제 확인을하셨죠.
4. [09:15:57] 문자 스트림을 어제 가만히 생각해보면 코드가 바이트 단위는 캐릭터 셋을 지정하는 것을 둔감합니다.
5. [09:17:36] 어제 너무 많이 막 했지만, 실제 그냥 일반적인 파일 입출력이라든지.
6. [09:22:38] 저희 어제 이 메서드를 호출을 해서 디렉토리 탐색하는 거를 하셨고요.
7. [09:24:41] 그런데 속성 같은 경우에는 어제도 이제트리뷰트를 가지고 있는 인터페이스가 따로 있었죠.
8. [09:25:08] 어제 굳이 4번 해서 파일을 한 4개를 만들어 가지고 이벤트 핸들링 하는 방법을 기존에 OP에 쓸 때 사용했던 문법과 접목을 해서 1번은 인터페이스 상속받고 하나는 인터페이스를 상속받은 클래스 구형 객체를 익스텐즈를 세 번째는 내가 또 커스텀 클래스를 만들어 가지고 익스텐즈를 했고 네 번째는 얘가 가지고 있는 자체를 후성 클래스 객체인 심플팔 디지털을 이용을 해서 사용을 했고 그거를 어제 했던 내용을 한번 정리를 해보면 1번 오버로드 메소드를 통해서 디렉토리 순환 결과 이벤트 파일을 확인하는 방법이 얘입니다.
9. [09:25:49] 파일비지털 인터페이스를 사용한 커스텀 파일비지털을 생성하는 방법을 어제 학습하셨습니다.
10. [09:26:37] 제어를 이렇게 할 수 있는 메소드들을 제공을 해줘 포스트나 프리오더나 이렇게 해서 어제 뭐 비제터라고 하는 일단 서픽스 메소드를 이용을 해서 일단은 만들어 가지고 얘네가 제공을 해주고 페일도 나면 이제 제공해주고 전체 4개의 메소드를 오버라이드를 하는 객체로 만들어 가지고 하나는 인터페이스 하나는 밑에 있는 심플 파일 비지터 클래스가 구현을 해서 우리가 사용할 수 있게끔 만들어줬습니다.
11. [09:28:12] 베이직 파일 어트리뷰트스라고 하는 것은 이게 어제 뭐가 있었어요.

**연결 흐름 요약**
강사는 전날 학습한 Java의 입출력(I/O, NIO.2), 스트림, `FileVisitor`를 활용한 디렉토리 탐색 및 이벤트 핸들링 등 실습 내용을 반복적으로 상기시키고 있습니다. 이는 어제 다룬 여러 핵심 개념을 체계적으로 재정리하여, 오늘 이어질 강의의 기반을 다지는 연결고리 역할을 합니다.

---

### gemini-3.5-flash  (15.9s)

**추출 문장 목록**

1. [09:15:05] 그런데 어제 저희는 네트워크를 하지 않았으니까 파일이 입출력 단위만 지금 학습을 하셨는데 바이트 스트림 단위 바이트 단위의 바이트 스트림 자체가 모든 팔 유형을 아우를 수가 있다고 생각하시면 되고 그래서 NIO2에서는 파일즈 가 가지고 있는 뉴 아풋스트림 그 다음에 뉴 인풋스트림 이렇게 되면 바이트 단위로 사용할 수 있는 요서드이죠.
2. [09:15:22] 자바가 가지고 있는 I이O 패키지는 클래스를 선택을 데이터 유형에 따라서 사용하는 거고, 그다음에 NIO2 같은 경우에는 메서드를 대상으로 저희가 데이터 프로세싱을 선택적으로 사용을 하는 그런 구문을 어제 확인을하셨죠.
3. [09:22:38] 저희 어제 이 메서드를 호출을 해서 디렉토리 탐색하는 거를 하셨고요.
4. [09:25:08] 어제 굳이 4번 해서 파일을 한 4개를 만들어 가지고 이벤트 핸들링 하는 방법을 기존에 OP에 쓸 때 사용했던 문법과 접목을 해서 1번은 인터페이스 상속받고 하나는 인터페이스를 상속받은 클래스 구형 객체를 익스텐즈를 세 번째는 내가 또 커스텀 클래스를 만들어 가지고 익스텐즈를 했고 네 번째는 얘가 가지고 있는 자체를 후성 클래스 객체인 심플팔 디지털을 이용을 해서 사용을 했고 그거를 어제 했던 내용을 한번 정리를 해보면 1번 오버로드 메소드를 통해서 디렉토리 순환 결과 이벤트 파일을 확인하는 방법이 얘입니다.

**연결 흐름 요약**
어제 학습한 파일 입출력 단위(바이트 스트림)의 특징과 NIO2의 데이터 프로세싱 선택 방식 및 디렉토리 탐색 메소드를 상기시킵니다. 이를 바탕으로 오늘 강의에서는 스트림 API와 연계한 가공 연산 방법 및 4가지 객체 지향 문법을 접목한 구체적인 이벤트 핸들링(심플 파일 비지터 활용 등)과 속성 제어 기법으로 확장하여 연결하고 있습니다.

---

### gemini-3.1-pro-preview  (24.3s)

**추출 문장 목록**
1. [09:15:05] 그런데 어제 저희는 네트워크를 하지 않았으니까 파일이 입출력 단위만 지금 학습을 하셨는데 바이트 스트림 단위 바이트 단위의 바이트 스트림 자체가 모든 팔 유형을 아우를 수가 있다고 생각하시면 되고 그래서 NIO2에서는 파일즈 가 가지고 있는 뉴 아풋스트림 그 다음에 뉴 인풋스트림 이렇게 되면 바이트 단위로 사용할 수 있는 요서드이죠.
2. [09:15:22] 자바가 가지고 있는 I이O 패키지는 클래스를 선택을 데이터 유형에 따라서 사용하는 거고, 그다음에 NIO2 같은 경우에는 메서드를 대상으로 저희가 데이터 프로세싱을 선택적으로 사용을 하는 그런 구문을 어제 확인을하셨죠.
3. [09:22:38] 저희 어제 이 메서드를 호출을 해서 디렉토리 탐색하는 거를 하셨고요.
4. [09:25:08] 어제 굳이 4번 해서 파일을 한 4개를 만들어 가지고 이벤트 핸들링 하는 방법을 기존에 OP에 쓸 때 사용했던 문법과 접목을 해서 1번은 인터페이스 상속받고 하나는 인터페이스를 상속받은 클래스 구형 객체를 익스텐즈를 세 번째는 내가 또 커스텀 클래스를 만들어 가지고 익스텐즈를 했고 네 번째는 얘가 가지고 있는 자체를 후성 클래스 객체인 심플팔 디지털을 이용을 해서 사용을 했고 그거를 어제 했던 내용을 한번 정리를 해보면 1번 오버로드 메소드를 통해서 디렉토리 순환 결과 이벤트 파일을 확인하는 방법이 얘입니다.
5. [09:25:49] 파일비지털 인터페이스를 사용한 커스텀 파일비지털을 생성하는 방법을 어제 학습하셨습니다.

**연결 흐름 요약**
강사는 전날 실습했던 파일 입출력 위주의 학습 내용, IO와 NIO2의 구문적 차이, 그리고 파일 4개를 직접 만들며 진행했던 이벤트 핸들링 경험을 구체적으로 상기시킵니다. 이를 바탕으로 오늘은 바이트 스트림의 원리, NIO2 기반의 디렉토리 탐색, 그리고 `FileVisitor` 인터페이스를 활용한 이벤트 처리 방식을 이론적으로 체계화하며 심화 학습으로 자연스럽게 이어가고 있습니다.

---

In [47]:
display(Markdown(
    f"**분석 구간** | 날짜: `{intro['date']}` · "
    f"`{intro['t_start']} ~ {intro['t_end']}` ({intro['row_count']}문장)\n\n---"
))

for r in results:
    header = f"### {r['model']}  ({r['elapsed']:.1f}s)"
    if r['error']:
        body = f"❌ **오류**: `{r['error']}`"
    else:
        body = r['text']
    display(Markdown(f"{header}\n\n{body}\n\n---"))

**분석 구간** | 날짜: `2026-02-03` · `09:11:04 ~ 09:40:57` (102문장)

---

### gemini-2.5-flash  (38.8s)

**추출 문장 목록**
1.  [09:12:15] 저희 어제 액션 11번의 노티스 파일에 내용은 적었지만 그래도 스트링 기반이고 얘네들은 데이터가 순차 이동이에요.
2.  [09:15:05] 그런데 어제 저희는 네트워크를 하지 않았으니까 파일이 입출력 단위만 지금 학습을 하셨는데 바이트 스트림 단위 바이트 단위의 바이트 스트림 자체가 모든 팔 유형을 아우를 수가 있다고 생각하시면 되고 그래서 NIO2에서는 파일즈 가 가지고 있는 뉴 아풋스트림 그 다음에 뉴 인풋스트림 이렇게 되면 바이트 단위로 사용할 수 있는 요서드이죠.
3.  [09:15:22] 자바가 가지고 있는 I이O 패키지는 클래스를 선택을 데이터 유형에 따라서 사용하는 거고, 그다음에 NIO2 같은 경우에는 메서드를 대상으로 저희가 데이터 프로세싱을 선택적으로 사용을 하는 그런 구문을 어제 확인을하셨죠.
4.  [09:22:38] 저희 어제 이 메서드를 호출을 해서 디렉토리 탐색하는 거를 하셨고요.
5.  [09:25:08] 어제 굳이 4번 해서 파일을 한 4개를 만들어 가지고 이벤트 핸들링 하는 방법을 기존에 OP에 쓸 때 사용했던 문법과 접목을 해서 1번은 인터페이스 상속받고 하나는 인터페이스를 상속받은 클래스 구형 객체를 익스텐즈를 세 번째는 내가 또 커스텀 클래스를 만들어 가지고 익스텐즈를 했고 네 번째는 얘가 가지고 있는 자체를 후성 클래스 객체인 심플팔 디지털을 이용을 해서 사용을 했고 그거를 어제 했던 내용을 한번 정리를 해보면 1번 오버로드 메소드를 통해서 디렉토리 순환 결과 이벤트 파일을 확인하는 방법이 얘입니다.
6.  [09:25:49] 파일비지털 인터페이스를 사용한 커스텀 파일비지털을 생성하는 방법을 어제 학습하셨습니다.
7.  [09:28:12] 베이직 파일 어트리뷰트스라고 하는 것은 이게 어제 뭐가 있었어요.

**연결 흐름 요약**
강의는 전날 학습한 파일 I/O의 기본 개념과 Java IO 및 NIO2의 차이점을 복습하며 시작합니다. 특히, 스트림 기반 데이터 처리 방식과 파일/디렉토리 탐색 및 이벤트 핸들링 기법, 그리고 `FileVisitor` 인터페이스를 활용한 구현 방법 등 어제 다룬 실습 내용을 상기시키며 오늘 강의의 심화된 파일 및 디렉토리 관리, 속성 활용 개념으로 자연스럽게 연결합니다.

---

### gemini-2.5-pro  (26.4s)

**추출 문장 목록**
1. [09:12:15] 저희 어제 액션 11번의 노티스 파일에 내용은 적었지만 그래도 스트링 기반이고 얘네들은 데이터가 순차 이동이에요.
2. [09:15:05] 그런데 어제 저희는 네트워크를 하지 않았으니까 파일이 입출력 단위만 지금 학습을 하셨는데 바이트 스트림 단위 바이트 단위의 바이트 스트림 자체가 모든 팔 유형을 아우를 수가 있다고 생각하시면 되고 그래서 NIO2에서는 파일즈 가 가지고 있는 뉴 아풋스트림 그 다음에 뉴 인풋스트림 이렇게 되면 바이트 단위로 사용할 수 있는 요서드이죠.
3. [09:15:22] 자바가 가지고 있는 I이O 패키지는 클래스를 선택을 데이터 유형에 따라서 사용하는 거고, 그다음에 NIO2 같은 경우에는 메서드를 대상으로 저희가 데이터 프로세싱을 선택적으로 사용을 하는 그런 구문을 어제 확인을하셨죠.
4. [09:15:57] 문자 스트림을 어제 가만히 생각해보면 코드가 바이트 단위는 캐릭터 셋을 지정하는 것을 둔감합니다.
5. [09:17:36] 어제 너무 많이 막 했지만, 실제 그냥 일반적인 파일 입출력이라든지.
6. [09:22:38] 저희 어제 이 메서드를 호출을 해서 디렉토리 탐색하는 거를 하셨고요.
7. [09:25:08] 어제 굳이 4번 해서 파일을 한 4개를 만들어 가지고 이벤트 핸들링 하는 방법을 기존에 OP에 쓸 때 사용했던 문법과 접목을 해서 1번은 인터페이스 상속받고 하나는 인터페이스를 상속받은 클래스 구형 객체를 익스텐즈를 세 번째는 내가 또 커스텀 클래스를 만들어 가지고 익스텐즈를 했고 네 번째는 얘가 가지고 있는 자체를 후성 클래스 객체인 심플팔 디지털을 이용을 해서 사용을 했고 그거를 어제 했던 내용을 한번 정리를 해보면 1번 오버로드 메소드를 통해서 디렉토리 순환 결과 이벤트 파일을 확인하는 방법이 얘입니다.
8. [09:25:49] 파일비지털 인터페이스를 사용한 커스텀 파일비지털을 생성하는 방법을 어제 학습하셨습니다.
9. [09:28:12] 베이직 파일 어트리뷰트스라고 하는 것은 이게 어제 뭐가 있었어요.

**연결 흐름 요약**
강사는 '어제'라는 키워드를 반복적으로 사용하여 전날 학습한 자바 IO 및 NIO2 관련 실습 내용(파일 입출력, 디렉토리 탐색, 이벤트 핸들링 등)을 상기시킵니다. 이를 바탕으로 당일 강의에서는 어제의 실습 내용을 이론적으로 심화하고, 각 기술의 특징과 사용법을 체계적으로 정리하며 개념을 확장하고 있습니다.

---

### gemini-3.5-flash  (18.9s)

**추출 문장 목록**

1. [09:12:15] 저희 어제 액션 11번의 노티스 파일에 내용은 적었지만 그래도 스트링 기반이고 얘네들은 데이터가 순차 이동이에요.
2. [09:14:53] 그런데 어제 저희는 네트워크를 하지 않았으니까 파일이 입출력 단위만 지금 학습을 하셨는데 바이트 스트림 단위 바이트 단위의 바이트 스트림 자체가 모든 팔 유형을 아우를 수가 있다고 생각하시면 되고 그래서 NIO2에서는 파일즈 가 가지고 있는 뉴 아풋스트림 그 다음에 뉴 인풋스트림 이렇게 되면 바이트 단위로 사용할 수 있는 요서드이죠.
3. [09:15:22] 자바가 가지고 있는 I이O 패키지는 클래스를 선택을 데이터 유형에 따라서 사용하는 거고, 그다음에 NIO2 같은 경우에는 메서드를 대상으로 저희가 데이터 프로세싱을 선택적으로 사용을 하는 그런 구문을 어제 확인을하셨죠.
4. [09:15:46] 문자 스트림을 어제 가만히 생각해보면 코드가 바이트 단위는 캐릭터 셋을 지정하는 것을 둔감합니다.
5. [09:17:41] 어제 너무 많이 막 했지만, 실제 그냥 일반적인 파일 입출력이라든지.
6. [09:22:38] 저희 어제 이 메서드를 호출을 해서 디렉토리 탐색하는 거를 하셨고요.
7. [09:24:41] 그런데 속성 같은 경우에는 어제도 이제트리뷰트를 가지고 있는 인터페이스가 따로 있었죠.
8. [09:25:08] 어제 굳이 4번 해서 파일을 한 4개를 만들어 가지고 이벤트 핸들링 하는 방법을 기존에 OP에 쓸 때 사용했던 문법과 접목을 해서 1번은 인터페이스 상속받고 하나는 인터페이스를 상속받은 클래스 구형 객체를 익스텐즈를 세 번째는 내가 또 커스텀 클래스를 만들어 가지고 익스텐즈를 했고 네 번째는 얘가 가지고 있는 자체를 후성 클래스 객체인 심플팔 디지털을 이용을 해서 사용을 했고 그거를 어제 했던 내용을 한번 정리를 해보면 1번 오버로드 메소드를 통해서 디렉토리 순환 결과 이벤트 파일을 확인하는 방법이 얘입니다.
9. [09:25:49] 파일비지털 인터페이스를 사용한 커스텀 파일비지털을 생성하는 방법을 어제 학습하셨습니다.
10. [09:28:12] 베이직 파일 어트리뷰트스라고 하는 것은 이게 어제 뭐가 있었어요.

---

**연결 흐름 요약**
어제 학습했던 자바의 스트림 기반 IO 패키지 및 NIO2의 파일 입출력, 디렉토리 탐색(속성 정의 및 메서드 활용)의 기초 원리를 상기시킵니다. 이를 바탕으로 오늘 강의에서는 NIO2의 성능적 우수성을 확인하고, `walkFileTree` 메서드와 `SimpleFileVisitor` 클래스를 활용한 이벤트 핸들링 및 디렉토리 탐색 오류 제어 등의 심화 구조 설계로 내용을 확장하여 연결하고 있습니다.

---

### gemini-3.1-pro-preview  (24.2s)

**추출 문장 목록**
1. [09:15:05] 그런데 어제 저희는 네트워크를 하지 않았으니까 파일이 입출력 단위만 지금 학습을 하셨는데 바이트 스트림 단위 바이트 단위의 바이트 스트림 자체가 모든 팔 유형을 아우를 수가 있다고 생각하시면 되고 그래서 NIO2에서는 파일즈 가 가지고 있는 뉴 아풋스트림 그 다음에 뉴 인풋스트림 이렇게 되면 바이트 단위로 사용할 수 있는 요서드이죠.
2. [09:15:22] 자바가 가지고 있는 I이O 패키지는 클래스를 선택을 데이터 유형에 따라서 사용하는 거고, 그다음에 NIO2 같은 경우에는 메서드를 대상으로 저희가 데이터 프로세싱을 선택적으로 사용을 하는 그런 구문을 어제 확인을하셨죠.
3. [09:25:08] 어제 굳이 4번 해서 파일을 한 4개를 만들어 가지고 이벤트 핸들링 하는 방법을 기존에 OP에 쓸 때 사용했던 문법과 접목을 해서 1번은 인터페이스 상속받고 하나는 인터페이스를 상속받은 클래스 구형 객체를 익스텐즈를 세 번째는 내가 또 커스텀 클래스를 만들어 가지고 익스텐즈를 했고 네 번째는 얘가 가지고 있는 자체를 후성 클래스 객체인 심플팔 디지털을 이용을 해서 사용을 했고 그거를 어제 했던 내용을 한번 정리를 해보면 1번 오버로드 메소드를 통해서 디렉토리 순환 결과 이벤트 파일을 확인하는 방법이 얘입니다.
4. [09:25:49] 파일비지털 인터페이스를 사용한 커스텀 파일비지털을 생성하는 방법을 어제 학습하셨습니다.

**연결 흐름 요약**
전날 실습 위주로 진행했던 파일 입출력, NIO2의 메서드 기반 데이터 프로세싱, 파일 비지터를 활용한 이벤트 핸들링 코드를 복습하며 환기시킵니다. 이를 바탕으로 바이트/문자 스트림의 이론적 특징을 체계적으로 정리하고, 오늘 본격적으로 다룰 디렉토리 탐색과 오버라이드 메소드를 통한 이벤트 제어 개념으로 자연스럽게 연결하고 있습니다.

---

## 5. 요약 테이블

In [14]:
import pandas as pd

summary = pd.DataFrame([
    {
        '모델'        : r['model'],
        '응답시간(s)' : round(r['elapsed'], 2),
        '응답길이(자)': len(r['text']) if r['text'] else 0,
        '오류'        : r['error'] or '-',
    }
    for r in results
])

display(summary)

,모델,응답시간(s),응답길이(자),오류
0,gemini-2.5-flash,7.59,235,-
1,gemini-2.5-pro,18.89,330,-
2,gemini-3.5-flash,7.01,355,-
3,gemini-3.1-pro-preview,12.98,275,-


## 6. [비교] Approach 2 — 입력: 인덱스 리스트 / 출력: 인덱스 번호

| | Approach 1 | Approach 2 |
|---|---|---|
| **LLM 입력** | `[HH:MM:SS] 문장` 형태의 timestamped text | `인덱스: 문장` 형태의 번호 붙은 리스트 |
| **LLM 출력** | 문장 원문 복사 | 인덱스 번호만 반환 |
| **장점** | 출력에서 바로 문장 확인 가능 | 출력 토큰 절감, hallucination 위험 낮음, 검증 용이 |

In [15]:
def load_intro_list(path, date=None, minutes=30):
    """kss csv에서 첫 N분 발화를 인덱스 부여 리스트로 반환한다."""
    rows = list(csv.DictReader(path.open(encoding='utf-8')))
    if date is None:
        date = sorted({r['date'] for r in rows})[0]
    date_rows = [r for r in rows if r['date'] == date]

    def to_sec(ts):
        h, m, s = map(int, ts.split(':'))
        return h * 3600 + m * 60 + s

    start_sec  = to_sec(date_rows[0]['timestamp'])
    cutoff_sec = start_sec + minutes * 60
    intro = [r for r in date_rows if start_sec <= to_sec(r['timestamp']) <= cutoff_sec]

    sentences = [{'idx': i, 'ts': r['timestamp'], 'text': r['text_raw']}
                 for i, r in enumerate(intro)]
    indexed_lines = '\n'.join(f"{s['idx']}: {s['text']}" for s in sentences)

    return {
        'date'         : date,
        'sentences'    : sentences,        # 원문 복원용
        'indexed_lines': indexed_lines,    # LLM 입력용
        'row_count'    : len(sentences),
    }

intro_v2 = load_intro_list(KSS_PATH, date="2026-02-02")
print(f"문장 수: {intro_v2['row_count']}")
print("--- indexed_lines 미리보기 (앞 5줄) ---")
print('\n'.join(intro_v2['indexed_lines'].splitlines()[:5]))

문장 수: 192
--- indexed_lines 미리보기 (앞 5줄) ---
0: 여러분 오늘 수업 진행하도록 하겠습니다.
1: 저희가 이제 오늘 1차 잡바 마지막 날입니다.
2: 그래서 지난 시간에 제너릭 타입을 이용을 해서 커스텀 컬렉션을 활용한 크루드 방법을 학습 이해를 하고 그다음에 데이터 흐름 읽기 쓰기를 구현하는 날이에요.
3: 자바는 NIO 패키지가 있고 그 다음에 NIO2라는 패키지를 가지고 있는데, 이 NI2는 자바 하는 애들이에요.
4: 그래서 이걸 세 가지 섹션으로 나눠서 진행을 하고 있습니다.


In [16]:
PROMPT_TEMPLATE_V2 = """\
당신은 강의 전사 텍스트를 분석하는 교육 전문가입니다.

아래는 백엔드 개발 입문 강의의 도입부(시작 후 약 30분) 전사 텍스트입니다.
각 줄은 "인덱스: 발화 내용" 형식입니다.
이 강의는 여러 날에 걸쳐 이어지는 시리즈 강의입니다.

[도입부 전사 텍스트]
{indexed_lines}

---

위 텍스트에서 **전날 학습 내용을 간략히 복습하면서 오늘 강의 내용과 연결하는 문장(들)**의
인덱스 번호를 추출하세요.

추출 기준:
- "지난 시간에", "저번에", "어제", "앞에서" 등 이전 수업을 언급하는 표현이 포함된 문장
- 이전 내용을 상기시킨 뒤 오늘 학습할 주제나 목표로 이어지는 흐름
- 단순 인사말·출석 체크·행정 안내는 제외

[중요 규칙]
- 반드시 위 텍스트에 존재하는 인덱스 번호만 사용할 것
- 인덱스 번호는 0 이상 {max_idx} 이하의 정수여야 함
- 문장 원문을 출력하지 말 것 — 인덱스만 반환할 것

응답 형식:
**추출 인덱스 목록**
[3, 7, 12, ...]

**연결 흐름 요약** (2~3줄 — 전날 어떤 내용을 복습했고 오늘 무엇으로 이어지는지)

해당하는 문장이 없으면 "복습·연결 문장 없음"이라고만 답하세요.
"""

prompt_v2 = PROMPT_TEMPLATE_V2.replace('{indexed_lines}', intro_v2['indexed_lines']).replace('{max_idx}', str(intro_v2['row_count'] - 1))
print(f"프롬프트 길이: {len(prompt_v2)}자")

프롬프트 길이: 12180자


In [17]:
import sys, csv, asyncio, time
from pathlib import Path

sys.path.insert(0, '..')
from app.core.config import Settings
import google.generativeai as genai

settings = Settings(_env_file='../.env')
genai.configure(api_key=settings.api_key)

MODELS   = ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.5-flash", "gemini-3.1-pro-preview"]
KSS_PATH = Path("../data/processed/lectures_kss.csv")

async def call_model(model_name: str, prompt: str) -> dict:
    t0 = time.time()
    try:
        model    = genai.GenerativeModel(model_name)
        response = await asyncio.to_thread(model.generate_content, prompt)
        return {'model': model_name, 'text': response.text, 'elapsed': time.time() - t0, 'error': None}
    except Exception as e:
        return {'model': model_name, 'text': None, 'elapsed': time.time() - t0, 'error': str(e)}

async def run_all(models, prompt):
    return await asyncio.gather(*[call_model(m, prompt) for m in models])

results_v2 = await run_all(MODELS, prompt_v2)

for r in results_v2:
    status = f"OK ({r['elapsed']:.1f}s)" if not r['error'] else f"ERROR — {r['error'][:80]}"
    print(f"[{r['model']}] {status}")

[gemini-2.5-flash] OK (51.7s)
[gemini-2.5-pro] OK (25.8s)
[gemini-3.5-flash] OK (9.0s)
[gemini-3.1-pro-preview] OK (10.7s)


In [18]:
import re

def resolve_indices(response_text: str, sentences: list) -> list:
    """응답 첫 줄에서 숫자를 파싱해 원문 문장을 복원한다."""
    first_line = (response_text or '').splitlines()[0] if response_text else ''
    nums = re.findall(r'\d+', first_line)
    return [sentences[int(n)] for n in nums if int(n) < len(sentences)]

for r in results_v2:
    header = f"### {r['model']}  ({r['elapsed']:.1f}s)"
    if r['error']:
        body = f"❌ **오류**: `{r['error']}`"
    else:
        resolved = resolve_indices(r['text'], intro_v2['sentences'])
        resolved_md = '\n'.join(
            f"{s['idx']}. [{s['ts']}] {s['text']}" for s in resolved
        )
        body = r['text'] + "\n\n**인덱스 복원 결과**\n" + resolved_md
    display(Markdown(f"{header}\n\n{body}\n\n---"))

### gemini-2.5-flash  (51.7s)

**추출 인덱스 목록**
[2, 32, 54, 64, 70, 72, 74, 84, 95, 99, 102, 108, 110, 153, 158, 161, 170, 190]

**연결 흐름 요약**
강의는 "지난 시간"에 학습한 제너릭 타입, 커스텀 컬렉션 활용 CRUD 방법을 복습하며 오늘 학습할 데이터 흐름 읽기/쓰기 구현으로 연결합니다.
또한, "지난주 어제 금요일"에 다룬 컬렉션 프레임워크와 "지난번에" 다룬 리스트 생성 방법 및 "수업 시간에 했던" 리버시드(reversed) 같은 구체적인 내용을 언급하며 현재 코어 라이브러리 탐색 및 자바 I/O, NIO, NIO2의 개념 설명으로 이어집니다. 학습 초기에는 튜토리얼을 보았지만, 이제 익숙해졌으니 코어 라이브러리를 활용하여 심화 학습을 진행할 것임을 상기시킵니다.

**인덱스 복원 결과**


---

### gemini-2.5-pro  (25.8s)

**추출 인덱스 목록**
[2, 70, 84, 99]

**연결 흐름 요약**
지난 시간에 학습한 제네릭과 커스텀 컬렉션(CRUD)을 언급하며, 이를 기반으로 오늘 배울 데이터 입출력(I/O)으로 주제를 자연스럽게 확장합니다. 또한, 새로운 I/O 개념을 학습하기 위해 공식 문서를 살펴보는 과정에서, 학생들이 이미 익숙한 '컬렉션 프레임워크'를 예시로 들어 문서 활용법을 안내하며 복습과 연결을 유도합니다.

**인덱스 복원 결과**


---

### gemini-3.5-flash  (9.0s)

**추출 인덱스 목록**
[2]

**연결 흐름 요약**
지난 시간에 제네릭 타입을 활용해 커스텀 컬렉션 기반의 CRUD 기능을 구현했던 학습 내용을 상기시킵니다. 이어서 오늘 수업에서는 그 연장선상에서 자바 IO 및 NIO 패키지를 사용해 데이터를 실제로 읽고 쓰는 '데이터 흐름'을 구현할 것임을 밝히며 자연스럽게 오늘 주제로 연결하고 있습니다.

**인덱스 복원 결과**


---

### gemini-3.1-pro-preview  (10.7s)

**추출 인덱스 목록**
[2]

**연결 흐름 요약**
지난 시간에 배운 제네릭 타입과 커스텀 컬렉션을 활용한 CRUD 구현 방법을 간략히 상기시킨 후, 이를 바탕으로 오늘 진행할 핵심 주제인 '데이터 흐름(입출력) 읽기 및 쓰기 구현'으로 자연스럽게 학습 목표를 연결하고 있습니다.

**인덱스 복원 결과**


---

In [22]:
import pandas as pd

summary_v2 = pd.DataFrame([
    {
        '모델'        : r['model'],
        '응답시간(s)' : round(r['elapsed'], 2),
        '응답길이(자)': len(r['text']) if r['text'] else 0,
        '오류'        : r['error'] or '-',
    }
    for r in results_v2
])
display(summary_v2)

,모델,응답시간(s),응답길이(자),오류
0,gemini-2.5-flash,51.70,406,-
1,gemini-2.5-pro,25.79,228,-
2,gemini-3.5-flash,9.05,195,-
3,gemini-3.1-pro-preview,10.72,159,-
